In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)

RAW_PATH = 'res/googleappstorev1.csv'
STEP2_PATH = 'output/step2_clean.csv'
STEP4_PATH = 'output/step4_size.csv'
STEP5_PATH = 'output/step5_price.csv'
STEP7_PATH = 'output/step7_type.csv'
STEP8_PATH = 'output/step8_content.csv'
STEP9_PATH = 'output/step9_category.csv'
APPSTORE_V14_PATH = 'output/appstorev1.4.csv'
FINAL_PATH = 'output/AppDataV2.csv'

os.makedirs('output', exist_ok=True)

## 缺失值查看

In [ ]:
raw_df = pd.read_csv(RAW_PATH, index_col=0)
print('原始数据形状:', raw_df.shape)
display(raw_df.head())
display(raw_df.isnull().any())
display(raw_df.isnull().sum())

## 缺失值处理 + 重复值处理

In [ ]:
df2 = pd.read_csv(RAW_PATH, index_col=0).copy()
df2['Rating'] = df2['Rating'].fillna(df2['Rating'].median())
df2 = df2.dropna().drop_duplicates().reset_index(drop=True)
df2.to_csv(STEP2_PATH)
print('Step2 输出:', STEP2_PATH, '形状=', df2.shape)
display(df2.isnull().sum())
df2.head()

## Rating 异常值观察

In [ ]:
df3 = pd.read_csv(STEP2_PATH, index_col=0)
display(df3[['Rating']].describe())
plt.figure(figsize=(8, 4))
df3['Rating'].hist(bins=20, color='steelblue', edgecolor='black')
plt.xlabel('Rating')
plt.ylabel('Frequency')
plt.title('Rating Distribution')
plt.show()

## Size 特征处理 + 异常值观察

In [ ]:
df4 = pd.read_csv(STEP2_PATH, index_col=0).copy()
size_raw = df4['Size'].astype(str).str.strip()
k_mask = size_raw.str.endswith('k')
m_mask = size_raw.str.endswith('M')
size_num = pd.Series(np.nan, index=df4.index, dtype='float64')
size_num.loc[m_mask] = pd.to_numeric(size_raw.loc[m_mask].str.replace('M', '', regex=False), errors='coerce')
size_num.loc[k_mask] = pd.to_numeric(size_raw.loc[k_mask].str.replace('k', '', regex=False), errors='coerce') / 1024
df4['Size'] = size_num
df4['Size'] = df4['Size'].fillna(df4.groupby('Category')['Size'].transform('mean'))
df4['Size'] = df4['Size'].fillna(df4['Size'].median()).round(3)
df4.to_csv(STEP4_PATH)
print('Step4 输出:', STEP4_PATH, '形状=', df4.shape)
display(df4['Size'].describe())
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df4, x='Size', y='Rating', s=12, alpha=0.5, color='orangered')
plt.title('Size vs Rating')
plt.show()

## Price 特征处理 + 异常值观察

In [ ]:
df5 = pd.read_csv(STEP4_PATH, index_col=0).copy()
df5['Price'] = pd.to_numeric(df5['Price'].astype(str).str.replace('$', '', regex=False), errors='coerce').fillna(0.0)
df5.to_csv(STEP5_PATH)
print('Step5 输出:', STEP5_PATH, '形状=', df5.shape)
display(df5['Price'].describe())
display(df5.loc[df5['Price'] > 300, ['App', 'Category', 'Rating', 'Reviews', 'Price']].head(10))
plt.figure(figsize=(7, 5))
sns.regplot(data=df5, x='Price', y='Rating', scatter_kws={'s': 12, 'alpha': 0.4}, line_kws={'color': 'black'})
plt.title('Price vs Rating')
plt.show()

## Installs + Reviews：特征处理、无量纲化、异常值处理

In [ ]:
df6 = pd.read_csv(STEP5_PATH, index_col=0).copy()
display(df6[['Installs', 'Reviews']].head())

df6['Installs'] = pd.to_numeric(
    df6['Installs'].astype(str).str.replace(',', '', regex=False).str.replace('+', '', regex=False),
    errors='coerce'
)
df6['Reviews'] = pd.to_numeric(df6['Reviews'], errors='coerce')
df6['Installs'] = df6['Installs'].fillna(df6['Installs'].median())
df6['Reviews'] = df6['Reviews'].fillna(df6['Reviews'].median())

def iqr_clip(s):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return s.clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

before_stats = df6[['Installs', 'Reviews']].describe()
df6['Installs'] = iqr_clip(df6['Installs'])
df6['Reviews'] = iqr_clip(df6['Reviews'])
after_stats = df6[['Installs', 'Reviews']].describe()
display(before_stats)
display(after_stats)

for col in ['Installs', 'Reviews']:
    log_col = f'{col}_log1p'
    norm_col = f'{col}_norm'
    df6[log_col] = np.log1p(df6[col])
    cmin, cmax = df6[log_col].min(), df6[log_col].max()
    df6[norm_col] = (df6[log_col] - cmin) / (cmax - cmin) if cmax > cmin else 0.0

display(df6[['Installs', 'Reviews', 'Installs_norm', 'Reviews_norm']].head())

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.boxplot(y=df6['Installs'], ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Installs (After IQR)')
sns.boxplot(y=df6['Reviews'], ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Reviews (After IQR)')
sns.histplot(df6['Installs_norm'], kde=True, ax=axes[1, 0], color='skyblue')
axes[1, 0].set_title('Installs_norm')
sns.histplot(df6['Reviews_norm'], kde=True, ax=axes[1, 1], color='salmon')
axes[1, 1].set_title('Reviews_norm')
plt.tight_layout()
plt.show()

appstore_v14 = df6[['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres']].copy()
appstore_v14.to_csv(APPSTORE_V14_PATH)
print('Step6 输出:', APPSTORE_V14_PATH, '形状=', appstore_v14.shape)

## Type 编码

In [ ]:
df7 = pd.read_csv(APPSTORE_V14_PATH, index_col=0).copy()
plt.figure(figsize=(5, 3))
df7['Type'].value_counts().plot(kind='bar', color=['#4C78A8', '#F58518'])
plt.title('Type Distribution')
plt.show()
df7['Type'] = df7['Type'].map({'Free': 1, 'Paid': 2}).astype('Int64')
df7.to_csv(STEP7_PATH)
print('Step7 输出:', STEP7_PATH, '形状=', df7.shape)
df7.head()

## Content Rating 处理与编码

In [ ]:
df8 = pd.read_csv(STEP7_PATH, index_col=0).copy()
display(df8['Content Rating'].value_counts())
display(df8[df8['Content Rating'] == 'Unrated'][['App', 'Category', 'Rating', 'Reviews', 'Installs']])
df8 = df8[df8['Content Rating'] != 'Unrated'].reset_index(drop=True)

plt.figure(figsize=(8, 4))
sns.boxplot(data=df8, x='Content Rating', y='Rating')
plt.xticks(rotation=45)
plt.title('Content Rating vs Rating')
plt.show()

content_order = {'Everyone': 0, 'Everyone 10+': 1, 'Teen': 2, 'Mature 17+': 3, 'Adults only 18+': 4}
df8['Content Rating'] = df8['Content Rating'].map(content_order).astype('Int64')
df8.to_csv(STEP8_PATH)
print('Step8 输出:', STEP8_PATH, '形状=', df8.shape)
df8.head()

## Category One-Hot 编码

In [ ]:
df9 = pd.read_csv(STEP8_PATH, index_col=0).copy()
display(df9['Category'].value_counts().head(10))
plt.figure(figsize=(18, 6))
sns.barplot(data=df9, x='Category', y='Rating', hue='Type', estimator=np.mean, errorbar=None)
plt.xticks(rotation=80)
plt.title('Category-Type-Rating')
plt.show()
df9 = pd.get_dummies(df9, columns=['Category'], dtype='int8')
df9.to_csv(STEP9_PATH)
print('Step9 输出:', STEP9_PATH, '形状=', df9.shape)
df9.head()

## Genres 编码 + 删除 App

In [ ]:
df10 = pd.read_csv(STEP9_PATH, index_col=0).copy()
df10['Genres'] = df10['Genres'].astype(str).apply(lambda x: x.split(';')[0].strip())
genre_map = {g: i for i, g in enumerate(sorted(df10['Genres'].unique()))}
df10['Genres'] = df10['Genres'].map(genre_map).astype('Int64')
df10 = df10.drop(columns=['App'])
df10.to_csv(FINAL_PATH)
print('最终输出:', FINAL_PATH, '形状=', df10.shape)
df10.head()